# Notebook 04 — Evaluation & Visualization
**Full ablation study, qualitative overlays, weather feature analysis**

Run **after** Notebooks 02 and 03 have produced their saved checkpoints and metric JSON files.

Contents:
1. Ablation study bar chart (Model A vs Model B)
2. Side-by-side prediction comparison
3. Failure case analysis
4. Weather feature permutation importance
5. Training curve overlay comparison

In [ ]:
import sys, os, json
os.chdir(r'd:\flood_segmentation')
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    %cd /content/drive/MyDrive/flood_segmentation
    %pip install -q rasterio tqdm scikit-learn

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from torch.utils.data import DataLoader

sys.path.insert(0, os.path.abspath('.'))
from src.datasets import build_datasets, WEATHER_FEATURES
from src.models   import get_model
from src.metrics  import compute_iou, compute_dice, compute_precision_recall_f1
from src.utils    import visualize_predictions, plot_ablation_results

sns.set_theme(style='whitegrid')
plt.rcParams.update({'figure.dpi': 120})

device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
FIG_DIR = 'results/figures'
os.makedirs(FIG_DIR, exist_ok=True)

IMG_SIZE    = 256
N_WEATHER   = len(WEATHER_FEATURES)
BATCH_SIZE  = 8
print(f'Device: {device}')

In [ ]:
# --- Load test metrics ---
with open('results/metrics_baseline.json') as f:    metrics_A = json.load(f)
with open('results/metrics_multimodal.json') as f:  metrics_B = json.load(f)

print('Model A — Baseline U-Net:')
for k, v in metrics_A.items(): print(f'  {k:12s}: {v:.4f}')
print('\nModel B — Weather-Aware U-Net:')
for k, v in metrics_B.items(): print(f'  {k:12s}: {v:.4f}')

In [ ]:
# --- 1. Ablation study bar chart ---
results = {
    'Model A\n(Baseline U-Net)':      metrics_A,
    'Model B\n(Weather-Aware U-Net)': metrics_B,
}
fig = plot_ablation_results(
    results,
    metrics=['iou', 'dice', 'precision', 'recall', 'f1'],
    save_path=f'{FIG_DIR}/04_ablation_barchart.png',
)
plt.show()

delta_iou  = metrics_B['iou']  - metrics_A['iou']
delta_dice = metrics_B['dice'] - metrics_A['dice']
print(f'\nΔ IoU  (B − A): {delta_iou:+.4f}')
print(f'Δ Dice (B − A): {delta_dice:+.4f}')

In [ ]:
# --- 2. Side-by-side predictions: Baseline vs Multimodal on same images ---
train_ds, val_ds, test_ds = build_datasets(
    data_dir='data/raw', weather_csv_path='data/processed/weather.csv',
    img_size=IMG_SIZE, train_ratio=0.70, val_ratio=0.15, seed=42,
)
loader = DataLoader(test_ds, batch_size=8, shuffle=False, num_workers=0)
imgs, weather, masks = next(iter(loader))

# Load both models from checkpoints
model_A = get_model('baseline', img_ch=2, base_ch=64)
ckpt_A  = torch.load('results/saved_models/baseline_unet_v2.pth', map_location=device)
model_A.load_state_dict(ckpt_A['model_state_dict'])
model_A = model_A.to(device).eval()

model_B = get_model('multimodal', img_ch=2, weather_dim=N_WEATHER, base_ch=64)
ckpt_B  = torch.load('results/saved_models/multimodal_unet_v2.pth', map_location=device)
model_B.load_state_dict(ckpt_B['model_state_dict'])
model_B = model_B.to(device).eval()

with torch.no_grad():
    preds_A = model_A(imgs.to(device)).cpu()
    preds_B = model_B(imgs.to(device), weather.to(device)).cpu()

print('Models loaded and predictions generated.')

In [ ]:
# Comparison grid: Image | GT | Baseline | Multimodal
n_show = 4
fig, axes = plt.subplots(n_show, 4, figsize=(16, 4 * n_show))
headers = ['Satellite (VV)', 'Ground Truth', 'Model A (Baseline)', 'Model B (Multimodal)']
for col, h in enumerate(headers): axes[0][col].set_title(h, fontsize=12, fontweight='bold')

for i in range(n_show):
    img_np = imgs[i, 0].numpy()           # VV channel
    gt     = masks[i, 0].numpy()
    pa     = (preds_A[i, 0] >= 0.5).float().numpy()
    pb     = (preds_B[i, 0] >= 0.5).float().numpy()

    iou_a  = compute_iou(preds_A[i:i+1], masks[i:i+1])
    iou_b  = compute_iou(preds_B[i:i+1], masks[i:i+1])

    axes[i][0].imshow(img_np, cmap='gray'); axes[i][0].axis('off')
    axes[i][1].imshow(gt,     cmap='Blues', vmin=0, vmax=1); axes[i][1].axis('off')
    axes[i][2].imshow(pa,     cmap='Blues', vmin=0, vmax=1)
    axes[i][2].set_xlabel(f'IoU={iou_a:.3f}', fontsize=10); axes[i][2].axis('off')
    axes[i][3].imshow(pb,     cmap='Blues', vmin=0, vmax=1)
    axes[i][3].set_xlabel(f'IoU={iou_b:.3f}', fontsize=10, color='green' if iou_b > iou_a else 'red')
    axes[i][3].axis('off')

plt.suptitle('Side-by-Side: Baseline vs Weather-Aware U-Net', fontsize=14, fontweight='bold')
plt.tight_layout()
fig.savefig(f'{FIG_DIR}/04_comparison_grid.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- 3. Failure case analysis: chips where baseline was worst ---
all_iou_A, all_iou_B = [], []
all_imgs, all_masks, all_preds_B = [], [], []

model_A.eval(); model_B.eval()
with torch.no_grad():
    for bt_imgs, bt_w, bt_masks in loader:
        p_a = model_A(bt_imgs.to(device)).cpu()
        p_b = model_B(bt_imgs.to(device), bt_w.to(device)).cpu()
        for bi in range(bt_imgs.size(0)):
            all_iou_A.append(compute_iou(p_a[bi:bi+1], bt_masks[bi:bi+1]))
            all_iou_B.append(compute_iou(p_b[bi:bi+1], bt_masks[bi:bi+1]))
        all_imgs.append(bt_imgs); all_masks.append(bt_masks); all_preds_B.append(p_b)

all_iou_A = np.array(all_iou_A)
all_iou_B = np.array(all_iou_B)
delta      = all_iou_B - all_iou_A

print(f'Mean IoU  — Baseline   : {all_iou_A.mean():.4f}')
print(f'Mean IoU  — Multimodal : {all_iou_B.mean():.4f}')
print(f'Chips improved by multimodal: {(delta > 0).sum()}/{len(delta)}')
print(f'Max improvement  : +{delta.max():.4f}  |  Max degradation: {delta.min():.4f}')

In [ ]:
# IoU distribution comparison plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(all_iou_A, bins=25, alpha=0.7, color='#3498db', label='Baseline')
axes[0].hist(all_iou_B, bins=25, alpha=0.7, color='#e74c3c', label='Multimodal')
axes[0].axvline(all_iou_A.mean(), color='#3498db', ls='--', lw=2)
axes[0].axvline(all_iou_B.mean(), color='#e74c3c', ls='--', lw=2)
axes[0].set(title='Test IoU Distribution', xlabel='IoU', ylabel='# Chips')
axes[0].legend()

axes[1].hist(delta, bins=25, color='#2ecc71', edgecolor='white', alpha=0.85)
axes[1].axvline(0, color='black', ls='--', lw=1.5)
axes[1].set(title='ΔIoU (Multimodal − Baseline) per Chip', xlabel='ΔIoU')

plt.tight_layout()
fig.savefig(f'{FIG_DIR}/04_iou_distributions.png', dpi=150)
plt.show()

In [ ]:
# --- 4. Weather feature permutation importance ---
# Measure how much model B's IoU drops when each weather feature is zeroed out.
print('Computing weather feature permutation importance...')
baseline_iou = all_iou_B.mean()
importance   = {}

for feat_idx, feat_name in enumerate(WEATHER_FEATURES):
    ious_permuted = []
    with torch.no_grad():
        for bt_imgs, bt_w, bt_masks in loader:
            bt_w_perm = bt_w.clone()
            bt_w_perm[:, feat_idx] = 0.0   # zero out this feature
            p = model_B(bt_imgs.to(device), bt_w_perm.to(device)).cpu()
            for bi in range(bt_imgs.size(0)):
                ious_permuted.append(compute_iou(p[bi:bi+1], bt_masks[bi:bi+1]))
    drop = baseline_iou - np.mean(ious_permuted)
    importance[feat_name] = drop
    print(f'  {feat_name:15s}: ΔIoU = {drop:+.4f}')

imp_df = pd.Series(importance).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
colors  = ['#e74c3c' if v > 0 else '#95a5a6' for v in imp_df.values]
imp_df.plot(kind='barh', ax=ax, color=colors, edgecolor='white')
ax.axvline(0, color='black', lw=1)
ax.set(title='Weather Feature Importance (Permutation Drop in IoU)',
       xlabel='ΔIoU when feature is zeroed', ylabel='')
plt.tight_layout()
fig.savefig(f'{FIG_DIR}/04_weather_importance.png', dpi=150)
plt.show()

In [ ]:
# --- 5. Summary table ---
summary = pd.DataFrame({
    'Metric':      ['IoU', 'Dice', 'Precision', 'Recall', 'F1', 'Loss'],
    'Baseline':    [metrics_A.get(k, float('nan')) for k in ['iou','dice','precision','recall','f1','loss']],
    'Multimodal':  [metrics_B.get(k, float('nan')) for k in ['iou','dice','precision','recall','f1','loss']],
})
summary['Δ']  = summary['Multimodal'] - summary['Baseline']
summary['▲']  = summary['Δ'].apply(lambda x: '▲' if x > 0 else ('▼' if x < 0 else '–'))
print('\n' + '='*55)
print('  Final Ablation Study — Test Set Results')
print('='*55)
print(summary.round(4).to_string(index=False))
summary.to_csv('results/ablation_summary.csv', index=False)
print('\nSaved: results/ablation_summary.csv')
print('\n✓ All evaluation plots saved to results/figures/')